# 🏥 HCC Deep Exploration Notebook

This notebook is a **complete exploration guide** for the `hccinfhir` library — the leading Python implementation of CMS risk adjustment models.

---
## 📚 Table of Contents
1. [Package Architecture](#architecture)
2. [Diagnosis Code Format Flexibility](#dx-codes)
3. [Available Models](#models)
4. [Detailed Score Breakdown](#breakdown)
5. [Advanced Demographics & Prefixes](#demographics)
6. [Disease Interactions](#interactions)
7. [Payment Adjustments (MACI, Normalization)](#payment)
8. [CMS Official vs hccinfhir (Comparison)](#comparison)
9. [Batch Processing (see hcc_batch_processor.py)](#batch)

---
## Package Architecture (How it Works Internally) <a id='architecture'></a>

When you call `calculate_from_diagnosis(...)`, it runs this internal pipeline:

```
Input DX Codes  ──▶  normalize (uppercase, remove dots)
                         │
                         ▼
               Dx → CC Mapping (ra_dx_to_cc_2026.csv)
               Look up each ICD-10 → Condition Category (CC)
                         │
                         ▼
               Age/Sex Edits (ra_dx_edits.csv)
               Apply CMS hardcoded edit rules (invalid/override)
                         │
                         ▼
               Hierarchies (ra_hierarchies_2026.csv)
               Apply hierarchical rules (parent HCC removes child HCC)
                         │
                         ▼
               Demographics Categorization
               age + sex + orec + dual_status → category (e.g., 'F65_69')
                         │
                         ▼
               Disease Interactions
               Identify special disease combos (e.g., DIABETES_HF_V28)
                         │
                         ▼
               Coefficients Lookup (ra_coefficients_2026.csv)
               Sum all applicable coefficients
                         │
                         ▼
               RAFResult with risk_score, hcc_list, details, etc.
```

In [ ]:
# Install / verify installation
!pip install hccinfhir tqdm pandas matplotlib
from hccinfhir import HCCInFHIR
print('hccinfhir loaded successfully!')

---
## 2. Diagnosis Code Format Flexibility <a id='dx-codes'></a>

**Key finding: `hccinfhir` accepts both dot-format and dotless formats!**

Internally, in `model_dx_to_cc.py`, line 41:
```python
dx = dx.upper().replace('.', '')
```
This means the library **strips dots and uppercases** ALL codes before lookup.

Supported input formats:

| Format | Example | Works? |
|--------|---------|--------|
| With dot (standard) | `E11.9` | ✅ Yes |
| Without dot | `E119` | ✅ Yes |
| Lowercase with dot | `e11.9` | ✅ Yes |
| Lowercase no dot | `e119` | ✅ Yes |
| With spaces | `E11 9` | ❌ No (space is not stripped) |

In [ ]:
from hccinfhir import HCCInFHIR
import pandas as pd

processor = HCCInFHIR(model_name='CMS-HCC Model V28')

# Test all formats for same codes
test_cases = {
    'With dots (standard)':    ['E11.9', 'I10', 'N18.3', 'I50.9'],
    'Without dots':            ['E119',  'I10', 'N183',  'I509'],
    'Lowercase with dots':     ['e11.9', 'i10', 'n18.3', 'i50.9'],
    'Lowercase without dots':  ['e119',  'i10', 'n183',  'i509'],
    'Mixed formats':           ['E119',  'i10', 'N18.3', 'i509'],
    'With spaces (broken)':    ['E11 9', 'N18 3'],   # This will NOT work
}

rows = []
for label, codes in test_cases.items():
    r = processor.calculate_from_diagnosis(codes, age=67, sex='F')
    rows.append({'Format': label, 'Codes': str(codes), 'Risk Score': r.risk_score, 'HCCs': r.hcc_list})

pd.DataFrame(rows).set_index('Format')

---
## 3. All Available Models <a id='models'></a>

The library supports 9 models across CMS-HCC (Medicare Advantage) and RxHCC (Part D Drug):

| Model Name | Use Case | Payment Year |
|------------|----------|--------------|
| `CMS-HCC Model V22` | Medicare Advantage (older) | 2014–2019 |
| `CMS-HCC Model V24` | Medicare Advantage (mid) | 2020–2023 |
| `CMS-HCC Model V28` | Medicare Advantage (current) | 2024–present |
| `CMS-HCC ESRD Model V21` | End-Stage Renal Disease | Dialysis patients |
| `CMS-HCC ESRD Model V24` | End-Stage Renal Disease (newer) | Dialysis patients |
| `RxHCC Model V08` | Medicare Part D Drug Risk | All sub-populations |
| `RxHCC Model V08 PDP_AND_MAPD` | Part D – Both plan types | Specific PDP/MAPD |
| `RxHCC Model V08 PDP_ONLY` | Part D – Standalone PDP only | Standalone drug plans |
| `RxHCC Model V08 MAPD_ONLY` | Part D – Medicare Advantage + Drug | MAPD plans |

**⚠️ Important:** V28 is fully phased in for 2024+ payment years. Use V28 unless you are doing historical analysis.

In [ ]:
from hccinfhir import HCCInFHIR
import pandas as pd

# Compare the same patient across different models
test_codes = ['E11.9', 'N18.3', 'I50.9']  # Diabetes, CKD Stage 3, Heart Failure
age, sex = 70, 'F'

community_models = [
    'CMS-HCC Model V22',
    'CMS-HCC Model V24',
    'CMS-HCC Model V28',
]

rows = []
for model in community_models:
    p = HCCInFHIR(model_name=model)
    r = p.calculate_from_diagnosis(test_codes, age=age, sex=sex)
    rows.append({
        'Model': model,
        'Risk Score': r.risk_score,
        'Demo Score': r.risk_score_demographics,
        'HCC Score': r.risk_score_hcc,
        'HCCs': ', '.join(r.hcc_list),
        'Category': r.demographics.category,
    })

df = pd.DataFrame(rows).set_index('Model')
print(f'Patient: Female, Age {age}, Codes: {test_codes}')
print()
df

---
## 4. Full Score Breakdown <a id='breakdown'></a>

The `RAFResult` object contains much more than just `risk_score`. Here's what every field means:

| Field | Description |
|-------|-------------|
| `risk_score` | Total RAF = demographics + all HCCs + interactions |
| `risk_score_demographics` | Score from age/sex category alone (no HCCs) |
| `risk_score_hcc` | Score from HCC conditions only (excl. demographics) |
| `risk_score_chronic_only` | Score from HCC conditions flagged as chronic only |
| `risk_score_payment` | Payment-adjusted score (after MACI + normalization) |
| `hcc_list` | List of active HCC category codes (after hierarchies applied) |
| `hcc_details` | HCC label, is_chronic flag, and coefficient for each HCC |
| `cc_to_dx` | Maps each CC/HCC to the ICD-10 codes that triggered it |
| `coefficients` | All coefficient name → value mappings used in score |
| `interactions` | Disease interaction flag dictionary |
| `demographics` | Full demographics object (category, disabled, fbd, etc.) |

In [ ]:
from hccinfhir import HCCInFHIR
import json

processor = HCCInFHIR(model_name='CMS-HCC Model V28')

# Patient with diabetes + heart failure (triggers disease interaction DIABETES_HF_V28)
codes = ['E119', 'N183', 'I509']   # dotless format - works perfectly
r = processor.calculate_from_diagnosis(codes, age=67, sex='F')

print('=== RISK SCORE COMPONENTS ===')
print(f'  Demographics only:    {r.risk_score_demographics:.4f}  (age/sex category = {r.demographics.category})')
print(f'  HCC conditions:       {r.risk_score_hcc:.4f}')
print(f'  Chronic only:         {r.risk_score_chronic_only:.4f}')
print(f'  TOTAL risk_score:     {r.risk_score:.4f}')
print()

print('=== APPLIED COEFFICIENTS ===')
for name, val in r.coefficients.items():
    print(f'  {name}: {val}')
print()

print('=== HCC DETAILS ===')
for d in r.hcc_details:
    print(f'  HCC {d.hcc:5s} | coeff={d.coefficient:.4f} | chronic={d.is_chronic} | {d.label}')
print()

print('=== DISEASE INTERACTIONS ===')
active = {k: v for k, v in r.interactions.items() if v > 0}
for k, v in active.items():
    print(f'  {k}: {v}')
print()

print('=== CC → DX MAPPING ===')
for cc, dx_set in r.cc_to_dx.items():
    print(f'  CC {cc} was triggered by: {dx_set}')
print()

print('=== DEMOGRAPHICS ===')
d = r.demographics
print(f'  Category:      {d.category}')
print(f'  Sex (internal):{d.sex}  (1=Male, 2=Female)')
print(f'  Non-Aged:      {d.non_aged}')
print(f'  Disabled:      {d.disabled}')
print(f'  Orig Disabled: {d.orig_disabled}')
print(f'  ESRD:          {d.esrd}')
print(f'  LTI:           {d.lti}')
print(f'  Full Dual:     {d.fbd}')
print(f'  Partial Dual:  {d.pbd}')

---
## 5. Advanced Demographics & Prefix Overrides <a id='demographics'></a>

The model uses demographic factors far beyond just age/sex. The key parameters are:

| Parameter | Values | Description |
|-----------|--------|-------------|
| `age` | 0–110+ | Patient age |
| `sex` | `M`/`F`/`1`/`2` | Patient sex |
| `dual_elgbl_cd` | `NA`,`00`–`10` | Dual Medicare/Medicaid eligibility |
| `orec` | `0`,`1`,`2`,`3` | Original reason for Medicare: 0=Aged, 1=Disabled, 2=ESRD, 3=DIB+ESRD |
| `crec` | `0`,`1`,`2`,`3` | Current reason for entitlement |
| `new_enrollee` | `True`/`False` | First year in Medicare |
| `snp` | `True`/`False` | Special Needs Plan |
| `lti` | `True`/`False` | Long-Term Institutionalized (nursing home) |
| `low_income` | `True`/`False` | Low Income Subsidy (RxHCC only) |
| `graft_months` | `1`,`2`,`3` | Months post-transplant (ESRD model only) |
| `prefix_override` | `DI_`,`CFA_`,`INS_`, etc. | Force a specific scoring segment |

### Dual Eligibility Codes (`dual_elgbl_cd`)
| Code | Meaning |
|------|---------|
| `NA` or `00` | Not dually eligible |
| `01`–`03` | Qualified Medicare Beneficiary (QMB) variations |
| `04`–`07` | Other dual categories |
| `08`–`10` | Full benefit dual eligible |

### Prefix Overrides (Force Scoring Segment)
| Prefix | Segment |
|--------|---------|
| `CNA_` | Community, Non-Dual, Aged |
| `CFA_` | Community, Full Benefit Dual, Aged |
| `CPA_` | Community, Partial Benefit Dual, Aged |
| `INS_` | Long-Term Institutionalized |
| `NE_`  | New Enrollee |
| `DI_`  | ESRD Dialysis |
| `DNE_` | ESRD Dialysis New Enrollee |

In [ ]:
from hccinfhir import HCCInFHIR
import pandas as pd

processor = HCCInFHIR(model_name='CMS-HCC Model V28')
codes = ['E119', 'N183']  # dotless - Diabetes + CKD

# Same patient scored under different demographic situations
scenarios = [
    {'name': 'Community, Non-Dual (default)', 'kwargs': {'age': 70, 'sex': 'F'}},
    {'name': 'Full Dual (Medicaid + Medicare)', 'kwargs': {'age': 70, 'sex': 'F', 'dual_elgbl_cd': '08'}},
    {'name': 'Partial Dual', 'kwargs': {'age': 70, 'sex': 'F', 'dual_elgbl_cd': '03'}},
    {'name': 'New Enrollee', 'kwargs': {'age': 65, 'sex': 'F', 'new_enrollee': True}},
    {'name': 'Disabled (age < 65, orec=1)', 'kwargs': {'age': 52, 'sex': 'F', 'orec': '1'}},
    {'name': 'Originally Disabled (aged in)', 'kwargs': {'age': 70, 'sex': 'F', 'orec': '1'}},
    {'name': 'Long-Term Institutionalized', 'kwargs': {'age': 70, 'sex': 'F', 'lti': True}},
    {'name': 'Force CFA_ prefix (override)', 'kwargs': {'age': 70, 'sex': 'F', 'prefix_override': 'CFA_'}},
]

rows = []
for s in scenarios:
    r = processor.calculate_from_diagnosis(codes, **s['kwargs'])
    rows.append({
        'Scenario': s['name'],
        'Category': r.demographics.category,
        'Demo Score': round(r.risk_score_demographics, 4),
        'HCC Score': round(r.risk_score_hcc, 4),
        'Total Score': round(r.risk_score, 4),
        'HCCs': ', '.join(r.hcc_list),
    })

pd.set_option('display.max_colwidth', 50)
pd.DataFrame(rows).set_index('Scenario')

---
## 6. Disease Interactions <a id='interactions'></a>

CMS-HCC V28 adds **extra coefficients when specific disease combinations appear together**. This is called a disease interaction. The interaction adds a bonus (or penalty) score on top of the individual HCC scores.

**Key interactions in V28:**

| Interaction | Triggered When | Coefficient (typical) |
|-------------|----------------|-----------------------|
| `DIABETES_HF_V28` | Diabetes (HCC 35–40) + Heart Failure (HCC 221–226) | ~+0.112 |
| `DIABETES_CHD_V28` | Diabetes + Coronary Heart Disease | varies |
| `DIABETES_CARD_RESP_FAIL_V28` | Diabetes + Cardiorespiratory Failure | varies |
| `HF_CHD_V28` | Heart Failure + Coronary Heart Disease | varies |

In [ ]:
from hccinfhir import HCCInFHIR

processor = HCCInFHIR(model_name='CMS-HCC Model V28')

# Show disease interaction bonus
r_diabetes = processor.calculate_from_diagnosis(['E119'], age=67, sex='F')                  # Diabetes alone
r_hf = processor.calculate_from_diagnosis(['I509'], age=67, sex='F')                        # Heart failure alone
r_both = processor.calculate_from_diagnosis(['E119', 'I509'], age=67, sex='F')              # Both together

sum_individual = r_diabetes.risk_score + r_hf.risk_score - r_diabetes.risk_score_demographics
interaction_bonus = r_both.risk_score - sum_individual

print(f'Diabetes alone (E11.9):          {r_diabetes.risk_score:.4f}')
print(f'Heart Failure alone (I50.9):     {r_hf.risk_score:.4f}')
print(f'Both together:                   {r_both.risk_score:.4f}')
print(f'Sum of individual (est):         {sum_individual:.4f}')
print(f'Interaction Bonus:               +{interaction_bonus:.4f}')
print()

print('Active interactions for diabetes + heart failure:')
for k, v in r_both.interactions.items():
    if 'DIABETES' in k or 'HF' in k:
        print(f'  {k}: {v}')
print()

print('All applied coefficients:')
for k, v in r_both.coefficients.items():
    print(f'  {k}: {v}')

---
## 7. Payment Adjustments (MACI & Normalization) <a id='payment'></a>

The **payment score** is different from the raw RAF score. CMS applies two adjustments:

| Parameter | Description | Formula Impact |
|-----------|-------------|----------------|
| `maci` | **M**edicare **A**dvantage **C**oding **I**ntensity Adjustment (0.0–1.0) | Reduces score: `score * (1 - maci)` |
| `norm_factor` | Normalization factor to account for model recalibration | Divides score: `/ norm_factor` |
| `frailty_score` | Added for frail elderly members in specific plans | Adds to final score |

**Payment formula:** `risk_score_payment = risk_score * (1 - maci) / norm_factor + frailty_score`

Typical real values (2026):
- MACI = **0.043** (4.3%)
- Norm factor = **~1.015** (varies by year and plan)

In [ ]:
from hccinfhir import HCCInFHIR
import pandas as pd

processor = HCCInFHIR(model_name='CMS-HCC Model V28')
codes = ['E119', 'N183', 'I509']
age, sex = 70, 'F'

scenarios = [
    {'name': 'Raw Score (no adjustments)', 'maci': 0.0, 'norm': 1.0, 'frailty': 0.0},
    {'name': 'With MACI 4.3%', 'maci': 0.043, 'norm': 1.0, 'frailty': 0.0},
    {'name': 'With MACI + Normalization 1.015', 'maci': 0.043, 'norm': 1.015, 'frailty': 0.0},
    {'name': 'With MACI + Norm + Frailty +0.2', 'maci': 0.043, 'norm': 1.015, 'frailty': 0.2},
]

rows = []
for s in scenarios:
    r = processor.calculate_from_diagnosis(codes, age=age, sex=sex,
                                           maci=s['maci'], norm_factor=s['norm'],
                                           frailty_score=s['frailty'])
    rows.append({
        'Scenario': s['name'],
        'Raw RAF': round(r.risk_score, 4),
        'Payment RAF': round(r.risk_score_payment, 4),
    })

pd.DataFrame(rows).set_index('Scenario')

---
## 8. CMS Official vs hccinfhir vs risk_adjustment_model <a id='comparison'></a>

| Feature | CMS Official (SAS) | hccinfhir (Python) | risk_adjustment_model |
|---------|-------------------|--------------------|-----------------------|
| **Language** | SAS | Python | Python |
| **License** | Free, public | MIT | MIT |
| **Requires SAS?** | ✅ Yes | ❌ No | ❌ No |
| **Python ≥ 3.12** | N/A | ❌ No (3.8+) | ✅ Required |
| **Models supported** | V22, V24, V28, ESRD, RxHCC | V22, V24, V28, ESRD, RxHCC | V24, V28 only |
| **FHIR input** | ❌ No | ✅ Yes (ExplanationOfBenefit) | ❌ No |
| **X12 837 claims** | ❌ No | ✅ Yes | ❌ No |
| **Dotless ICD-10** | ✅ Yes | ✅ Yes | Need to check |
| **Coefficients** | 2026 official | 2025, 2026, 2027 (proposed) | 2024, 2025 |
| **pip install** | N/A | `pip install hccinfhir` | `pip install risk_adjustment_model` (requires Python 3.12) |
| **Maintained** | CMS only | Actively | Actively |

### ⚠️ CMS Official SAS Software
CMS publishes the official software in SAS format at:
https://www.cms.gov/medicare/payment/medicare-advantage-rates-statistics/risk-adjustment

The files include:
- `V28I0ED.SAS` — ICD-10 to CC mapping and edits
- `V28I0H.SAS` — Hierarchies
- `V28I0P.SAS` — Scoring/coefficients

**`hccinfhir` implements the same logic from these SAS files in Python**, using the same reference CSV data from CMS.

In [ ]:
from hccinfhir import HCCInFHIR

# Side-by-side comparison of V24 vs V28 for same patient
# This reflects the real-world transition from V24 to V28 in 2024
processor_v24 = HCCInFHIR(model_name='CMS-HCC Model V24')
processor_v28 = HCCInFHIR(model_name='CMS-HCC Model V28')

test_patients = [
    {'name': 'Complex Diabetic Patient',   'codes': ['E1169', 'N183', 'I509', 'E1165'], 'age': 72, 'sex': 'F'},
    {'name': 'Cancer Patient',             'codes': ['C3490', 'J449'],                  'age': 68, 'sex': 'M'},
    {'name': 'Heart Disease Patient',      'codes': ['I509', 'I2510', 'I4891'],         'age': 78, 'sex': 'F'},
    {'name': 'Healthy Enrollee',           'codes': ['Z0000'],                           'age': 67, 'sex': 'M'},
]

print(f'{"Patient":<30} {"V24 Score":>10} {"V28 Score":>10} {"Change":>10}')
print('-' * 65)

for p in test_patients:
    r24 = processor_v24.calculate_from_diagnosis(p['codes'], age=p['age'], sex=p['sex'])
    r28 = processor_v28.calculate_from_diagnosis(p['codes'], age=p['age'], sex=p['sex'])
    change = r28.risk_score - r24.risk_score
    direction = '▲' if change > 0 else ('▼' if change < 0 else '=')
    print(f"{p['name']:<30} {r24.risk_score:>10.4f} {r28.risk_score:>10.4f} {direction}{abs(change):>9.4f}")

---
## 9. Using RxHCC Model (Part D Drug Risk) <a id='rxhcc'></a>

The **RxHCC model** is used for Medicare **Part D drug benefit** risk adjustment. It is separate from the CMS-HCC model.

Key differences from CMS-HCC:
- Uses `low_income` flag (Low Income Subsidy / Extra Help)
- Has different HCC categories (different set from CMS-HCC)
- Four sub-models: all, PDP_AND_MAPD, PDP_ONLY, MAPD_ONLY

In [ ]:
from hccinfhir import HCCInFHIR
import pandas as pd

codes = ['E119', 'N183', 'I509']
age, sex = 70, 'F'

rx_models = [
    ('RxHCC Model V08',             {'low_income': False}),
    ('RxHCC Model V08',             {'low_income': True}),
    ('RxHCC Model V08 PDP_AND_MAPD',{'low_income': True}),
    ('RxHCC Model V08 PDP_ONLY',    {'low_income': True}),
    ('RxHCC Model V08 MAPD_ONLY',   {'low_income': True}),
]

rows = []
for model_name, extra in rx_models:
    p = HCCInFHIR(model_name=model_name)
    r = p.calculate_from_diagnosis(codes, age=age, sex=sex, **extra)
    rows.append({
        'Model': model_name,
        'Low Income': extra.get('low_income', False),
        'Risk Score': round(r.risk_score, 4),
        'HCCs': ', '.join(r.hcc_list),
        'Category': r.demographics.category,
    })

pd.DataFrame(rows).set_index('Model')